# Attribute ML: quality prediction 

we load the dataset and we explore

In [ ]:
import sys
print(sys.executable)

In [ ]:

import pandas as pd
import numpy as np  
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn import metrics
from sklearn.svm import SVC

from sklearn.linear_model import LogisticRegression
import mlflow,mlflow.sklearn
from mlflow.models import infer_signature #it understands and saves the data columns and rows
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (mean_squared_error, r2_score, mean_absolute_error,roc_auc_score, accuracy_score, precision_score, recall_score, f1_score,classification_report, confusion_matrix)
import numpy as np
import optuna as opt #for hyperparameter tuning
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, KFold, cross_val_score
from xgboost import XGBRegressor
opt.logging.set_verbosity(opt.logging.WARNING) #to suppress optuna warnings
random_state = 42
np.random.seed(random_state)

mlflow.set_tracking_uri("sqlite:////home/nasia/wine-innovation-engine/notebooks/mlflow.db")  # ΠΡΩΤΑ URI
mlflow.set_experiment("wine_quality_prediction")   # ΜΕΤΑ experiment

In [ ]:

wine_dataset=pd.read_csv("/home/nasia/wine-innovation-engine/data/winequality-white (1).csv", sep=";")

wine_dataset

fixed acidity -> it gives strucure and taste
volatile acidity->vinegar if it increased -> bad quality
Residual sugar->we want lower because we want dry 
chlorides->salt,it changes the taste and it helps with microbial stability
Free SO₂+Total SO₂->antimicrobial and antioxidant ,we want ~200 mg/L,if it is increased ->cheap wine
Sulphates->taste
alcohol->high percentage->complete fermentation 

In [ ]:
wine_dataset.info()

In [ ]:
wine_dataset.isnull().sum()

In [ ]:
wine_dataset.describe()

In [ ]:
#visualisation
plt.figure(figsize=(10,6))
sns.histplot(data=wine_dataset, x='quality', bins=6, kde=True)

In [ ]:
#visualizing feauture relationships with quality
plt.figure(figsize=(10,6))
sns.heatmap(wine_dataset.corr(method='pearson'), annot=True, cmap='coolwarm', fmt='.2f')

In [ ]:
#we see that alcohol has a strong positive correlation with quality, while volatile acidity has a strong negative correlation with quality.
#density is correlated to sugar and alcohol and that is logical because density is a measure of how much mass is in a given volume, and sugar and alcohol are both substances that contribute to the mass of a liquid.(ethanol is decreasing it)


In [ ]:
#violin plot 
plt.figure(figsize=(10,6))
sns.violinplot(x='quality', y='alcohol', data=wine_dataset, palette='muted')
plt.title('Violin plot of Alcohol content by Quality')
plt.show()

In [ ]:
#drop duplicates :if we dont ->data leakage
wine_dataset = wine_dataset.drop_duplicates().reset_index(drop=True)

wine_dataset.info()

#we will use random forest as our main model so we are okay with the multicollinearity ,we will be carefull with the shap values

In [ ]:
#X/y separation
X=wine_dataset.drop('quality', axis=1)
y=wine_dataset['quality']

print(X.shape, y.shape)

In [ ]:
#SPLITING 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

1.ridge regression model with cross validation

In [ ]:

#we put everyhting in mlflow to track the model and its parameters,we use ridge
#we make the pipeline

#linear regression model
#we put everyhting in mlflow to track the model and its parameters,we use linear regression because we are predicting a continuous variable (quality) based on the features of the wine dataset.




with mlflow.start_run(run_name="ridge_regression_model_with_5_3_25"):
    mlflow.log_param("model_type", "ridge_regression")
    mlflow.log_param("dataset", "winequality-white.csv")
    mlflow.log_param("num_rows", wine_dataset.shape[0])
    mlflow.log_param("test_size", 0.2)
    mlflow.log_param("scaler", "StandardScaler") 

    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('ridge', Ridge()),
    ])

    param_grid = {"ridge__alpha": np.logspace(-5, 3, 25)}
    cv = KFold(n_splits=5, shuffle=True, random_state=random_state)

    grid = GridSearchCV(pipe, param_grid, cv=cv,
                         scoring='neg_root_mean_squared_error',
                         n_jobs=-1, refit=True)
    grid.fit(X_train, y_train)

    best_alpha = grid.best_params_["ridge__alpha"]
    cv_rmse = -grid.best_score_  

    mlflow.log_param("best_alpha", best_alpha)
    mlflow.log_metric("cv_rmse", cv_rmse)

    # test set 
    y_pred = grid.best_estimator_.predict(X_test)
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    test_r2 = r2_score(y_test, y_pred)

    mlflow.log_metric("test_rmse", test_rmse)
    mlflow.log_metric("test_r2", test_r2)

    mlflow.sklearn.log_model(grid.best_estimator_, name="ridge_regression_model")

    print("Best alpha:", best_alpha)
    print("CV RMSE:", cv_rmse)
    print("Test RMSE:", test_rmse)
    print("Test R²:", test_r2)

#Very small alpha so maybe the regularization does not help much here ,linear regression will do the same but we can keep it as a baseline
#we tried different alpha but the result was the same so multicollinearity in the dataset was not very serious to be fixed from the l2 shrinkage 


2.RANDOM FOREST WITH OPTUNA cross validation 



In [ ]:
#made the objective to call it enough times  THE INNER FOLD
 def objective(trial,X_train,y_train,inner_cv):#we give the inner split #trial recommends some hyperparameters, inner_cv is the cross-validation strategy
    params={ "n_estimators": trial.suggest_int("n_estimators", 200, 500, step=100),
        "max_depth": trial.suggest_int("max_depth", 3, 25),#we do not want very high because there is the danger of overfitting
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
        "max_features": trial.suggest_float("max_features", 0.3, 1.0),}


    pipe=Pipeline([
        ('scaler', StandardScaler()),
        ('model',RandomForestRegressor(random_state=random_state,n_jobs=-1, **params))#only here n_jobs=-1 because we want to use all cores for training the model, not for hyperparameter tuning
    ])

    scores=cross_val_score(
        pipe,X_train,y_train,cv=inner_cv,scoring='neg_root_mean_squared_error',) 
    
     
    return scores.mean()#the mean of the 5 folds ,how goood is the combination of the params of the inner split 


we split the xtest to 5 folds the outer ,these are 80% train and 20% test and the 80% we resplit it to 80% train and test 

In [ ]:
from sklearn.ensemble import RandomForestRegressor
import optuna

outer_cv = KFold(n_splits=5, shuffle=True, random_state=random_state)
outer_splits=list(outer_cv.split(X_train,y_train))#it freezes it 
inner_cv = KFold(n_splits=5, shuffle=True, random_state=random_state)

outer_test_scores = []
fold_best_params = []
#log with mlflow 


with mlflow.start_run(run_name="random_forest_nested_cv") as parent_run:
    mlflow.log_param("model_type", "random_forest")
    mlflow.log_param("n_outer_folds", 5)
    mlflow.log_param("n_inner_folds", 5)
    mlflow.log_param("n_trials_per_fold", 30)

    for fold_i, (train_idx, test_idx) in enumerate(outer_splits):
        X_outer_train, X_outer_test = X_train.iloc[train_idx], X_train.iloc[test_idx]
        y_outer_train, y_outer_test = y_train.iloc[train_idx], y_train.iloc[test_idx]
#TPE SAMPLER ->bayesian optimization 
        with mlflow.start_run(run_name=f"outer_fold_{fold_i}", nested=True):
            study = optuna.create_study(direction="maximize",sampler=optuna.samplers.TPESampler(seed=random_state),)

            study.optimize(
                lambda trial: objective(trial, X_outer_train, y_outer_train, inner_cv),
                n_trials=30,
            )
      
            best_params = study.best_params
            inner_cv_rmse = -study.best_value
      
            final_model = Pipeline([
                ("scaler", StandardScaler()),
                ("model", RandomForestRegressor(random_state=random_state, n_jobs=-1, **best_params)),
            ])
            final_model.fit(X_outer_train, y_outer_train)
            y_pred_outer = final_model.predict(X_outer_test)
            outer_test_rmse = np.sqrt(mean_squared_error(y_outer_test, y_pred_outer))

            
            



        
            mlflow.log_params(best_params)
            mlflow.log_metric("inner_cv_rmse", inner_cv_rmse)
            mlflow.log_metric("outer_test_rmse", outer_test_rmse)

            outer_test_scores.append(outer_test_rmse)
            fold_best_params.append(best_params)

            print(f"Fold {fold_i}: best_params={best_params}, outer_test_rmse={outer_test_rmse:.4f}")

    mean_rmse = np.mean(outer_test_scores)
    std_rmse = np.std(outer_test_scores)
    mlflow.log_metric("nested_rmse_mean", mean_rmse)
    mlflow.log_metric("nested_rmse_std", std_rmse)

    print(f"\nNested CV RMSE: {mean_rmse:.4f} ± {std_rmse:.4f}")

In [ ]:
#optuna study to the whole X_train set without outer split 
with mlflow.start_run(run_name="random_forest_final"):
    mlflow.log_param("model_type","random_forest_final")
    mlflow.log_param("n_trials",50)

    cv = KFold(n_splits=5, shuffle=True, random_state=random_state)

    study_rf=optuna.create_study(direction="maximize",sampler=optuna.samplers.TPESampler(seed=random_state))
#adjusting the cv
    

    study_rf.optimize(
    lambda trial: objective(trial, X_train, y_train, cv),
    n_trials=50,
)

    rf_best_params=study_rf.best_params
    cv_rmse=-study_rf.best_value

    #building the final model with the best params

    rf_final_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RandomForestRegressor(random_state=random_state, n_jobs=-1, **rf_best_params)),  
])
    rf_final_model.fit(X_train,y_train)

    #infer siganture
    signature=infer_signature(X_train,rf_final_model.predict(X_train))
    input_example=X_train.iloc[:5]#we give an example

    #evaluate in the x_test
    y_pred=rf_final_model.predict(X_test)
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    test_mae = mean_absolute_error(y_test, y_pred)
    test_r2 = r2_score(y_test, y_pred)

    # Log every result
    mlflow.log_params(rf_best_params)
    mlflow.log_metric("cv_rmse", cv_rmse)
    mlflow.log_metric("test_rmse", test_rmse)
    mlflow.log_metric("test_mae", test_mae)
    mlflow.log_metric("test_r2", test_r2)

    
    mlflow.sklearn.log_model(rf_final_model, name="random_forest_rf_final_model",signature=signature,input_example=input_example)
    #we will load to the streamlit with pyfunc

    print(f"Best params: {rf_best_params}")
    print(f"CV RMSE: {cv_rmse:.4f}")
    print(f"Test RMSE: {test_rmse:.4f}")
    print(f"Test MAE: {test_mae:.4f}")
    print(f"Test R²: {test_r2:.4f}")


we have better RSME IN THE final random forest model because it saw more data 

3.XGBOOST 

In [ ]:
import xgboost as xgb
print(xgb.__version__)

print(X_train.shape)

In [ ]:
#making an objective again,it scores the recipe of the hyperparameters
#we neeed to have the trial because optuna takes a function that takes trial and returns a number ,optuna finds the prices that maximize it 
def objective_xgb(trial, X_train, y_train, inner_cv):
    params = {
        "n_estimators":     trial.suggest_int("n_estimators", 200, 1000, step=100),
        "max_depth":        trial.suggest_int("max_depth", 3, 10),
        "learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "subsample":        trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_lambda":       trial.suggest_float("reg_lambda", 0.1, 10.0, log=True),
    }


    pipe=Pipeline([
        ("scaler",StandardScaler()),
         ("model",XGBRegressor(
             random_state=random_state,
             n_jobs=-1,
             objective="reg:squarederror",#this time the objective is the loss function
             tree_method="hist", #we need for the splitting ,it does binning Divides each feature equals population bins (quantiles). Then, candidate splits are only the boundaries of the bins
             **params
         ))
    ])

    scores=cross_val_score(pipe,X_train,y_train,cv=inner_cv,scoring="neg_root_mean_squared_error")

    return scores.mean()


    

In [ ]:
#smoke test to see if objective_xgb runs 
s=optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=random_state),
)

s.optimize(lambda t: objective_xgb(t,X_train,y_train,inner_cv),n_trials=5)
print("smoke RMSE:", -s.best_value)

In [ ]:
# XGBoost nested CV 
outer_test_scores_xgb = []
fold_best_params_xgb = []

with mlflow.start_run(run_name="xgb_nested_cv") as parent_run:
    mlflow.log_param("model_type", "xgb")
    mlflow.log_param("n_outer_folds", 5)
    mlflow.log_param("n_inner_folds", 5)
    mlflow.log_param("n_trials_per_fold", 30)

    for fold_i, (train_idx, test_idx) in enumerate(outer_splits):
        X_outer_train, X_outer_test = X_train.iloc[train_idx], X_train.iloc[test_idx]
        y_outer_train, y_outer_test = y_train.iloc[train_idx], y_train.iloc[test_idx]

        with mlflow.start_run(run_name=f"outer_fold_{fold_i}", nested=True):

            # --- INNER hyperparameters ---
            study = optuna.create_study(
                direction="maximize",
                sampler=optuna.samplers.TPESampler(seed=random_state),
            )
            study.optimize(
                lambda trial: objective_xgb(trial, X_outer_train, y_outer_train, inner_cv),
                n_trials=30,
            )

            best_params = study.best_params
            inner_cv_rmse = -study.best_value

            # --- OUTER:
            final_model = Pipeline([
                ("scaler", StandardScaler()),
                ("model", XGBRegressor(
                    random_state=random_state,
                    n_jobs=-1,
                    objective="reg:squarederror",
                    tree_method="hist",
                    **best_params
                )),
            ])
            final_model.fit(X_outer_train, y_outer_train)
            y_pred_outer = final_model.predict(X_outer_test)
            outer_test_rmse = np.sqrt(mean_squared_error(y_outer_test, y_pred_outer))

            mlflow.log_params(best_params)
            mlflow.log_metric("inner_cv_rmse", inner_cv_rmse)
            mlflow.log_metric("outer_test_rmse", outer_test_rmse)

            outer_test_scores_xgb.append(outer_test_rmse)
            fold_best_params_xgb.append(best_params)

            print(f"Fold {fold_i}: outer_test_rmse={outer_test_rmse:.4f}  {best_params}")

    mean_rmse = np.mean(outer_test_scores_xgb)
    std_rmse = np.std(outer_test_scores_xgb)
    mlflow.log_metric("nested_rmse_mean", mean_rmse)
    mlflow.log_metric("nested_rmse_std", std_rmse)

    print(f"\nXGBoost nested CV RMSE: {mean_rmse:.4f} ± {std_rmse:.4f}")

In [ ]:
#final optuna study,fit on the whole train ,predict the y test
#final optuna study,fit on the whole train ,predict the y test

with mlflow.start_run(run_name="xgb_final_model"):
    mlflow.log_param("model_type","xgb_final_model")
    mlflow.log_param("n_trials",50)
#to split the whole dataset again on 5 folds
    cv=KFold(n_splits=5,shuffle=True,random_state=random_state)
#we maximize because we want the scoring big neg_root_mean_squared_error
    study_xgb=optuna.create_study(direction="maximize",sampler=optuna.samplers.TPESampler(seed=random_state))

    study_xgb.optimize(
        lambda trial: objective_xgb(trial,X_train,y_train,cv),
        n_trials=50,
    )

    xgb_best_params=study_xgb.best_params
    cv_rmse=-study_xgb.best_value


    xgb_final_model=Pipeline([
        ("scaler",StandardScaler()),
        ("model", XGBRegressor(random_state=random_state, n_jobs=-1, **xgb_best_params))
    ])

    xgb_final_model.fit(X_train,y_train)

    #infer siganture
    signature=infer_signature(X_train,xgb_final_model.predict(X_train))
    input_example=X_train.iloc[:5]

    #evaluate in the x_test
    y_pred=xgb_final_model.predict(X_test)
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    test_mae = mean_absolute_error(y_test, y_pred)
    test_r2 = r2_score(y_test, y_pred)

    # Log every result
    mlflow.log_params(xgb_best_params)
    mlflow.log_metric("cv_rmse", cv_rmse)
    mlflow.log_metric("test_rmse", test_rmse)
    mlflow.log_metric("test_mae", test_mae)
    mlflow.log_metric("test_r2", test_r2)


In [ ]:
print(f"Best params: {xgb_best_params}")
print(f"CV RMSE: {cv_rmse:.4f}")
print(f"Test RMSE: {test_rmse:.4f}")
print(f"Test MAE: {test_mae:.4f}")
print(f"Test R²: {test_r2:.4f}")

Lets compare the two models 
We saw that in the two models the folds 3 and 4 (are the same folds because we have froze them) have significant higher RMSE that indicates that these two folds have bigger variance so we do a paired test to remove the fold effect

In [ ]:
from scipy import stats

#we have them in arrays
rf=np.array(outer_test_scores)
xgb=np.array(outer_test_scores_xgb)

d=rf-xgb

print("corr between folds:",np.corrcoef(rf,xgb))

print("differences:", d, " mean:", d.mean())
#rel=related samples ,it runs a one-sample t-test with null hypothesis
t,p=stats.ttest_rel(rf,xgb)

print(f"paired t = {t:.3f}, p = {p:.4f}")

there is no significant difference but we choose random forest because less parameters ,faster 

lets start with the shap values 

In [ ]:
import shap
import time
shap.__version__

In [ ]:
#we write again the best params 
rf_best_params = study_final.best_params

In [ ]:
assert "learning_rate" not in rf_best_params, "XGBoost params in RF variable!"
assert "n_estimators" in rf_best_params

from sklearn.ensemble import RandomForestRegressor


rf_unscaled = RandomForestRegressor(
    random_state=random_state,
    n_jobs=-1,
    **rf_best_params,
)
rf_unscaled.fit(X_train, y_train)

pred_unscaled = rf_unscaled.predict(X_test)
pred_scaled   = rf_final_model.predict(X_test)


print("max |Δ| vs scaled RF pipeline:", np.abs(pred_unscaled - pred_scaled).max())
print("RMSE:", np.sqrt(mean_squared_error(y_test, pred_unscaled)))
print("R²  :", r2_score(y_test, pred_unscaled))

not much of a difference between the scaled and unscaled 

In [ ]:
#having a background to comapre 
background = shap.utils.sample(X_train, 100, random_state=random_state)

explainer = shap.TreeExplainer(rf_unscaled, data=background)

print("expected_value:", explainer.expected_value)
print("mean pred on background:", rf_unscaled.predict(background).mean())  
print("y_train mean:", y_train.mean())
#calculating the time it takes to explain 50 samples and estimating the time for the whole test set
t0 = time.time()
_ = explainer(X_test.iloc[:50])
dt = time.time() - t0
print(f"{dt:.1f}s for  50 samples  {dt/50*len(X_test):.0f}s για όλο το test set")

In [ ]:
#keeping the time 

t0=time.time()
sv=explainer(X_test,check_additivity=True)
print(f"{time.time()-t0:.0f}s for {len(X_test)} samples")

#the explainer has : values+base_values+all together samples
print("values:",sv.values.shape)
print("base_values:", np.unique(sv.base_values).round(4))


#lets do the additivity check
recon=sv.values.sum(axis=1)+sv.base_values
print("max additivity error:", np.abs(recon - rf_unscaled.predict(X_test)).max())

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SHAP_DIR = PROJECT_ROOT / "results" / "phase4_shap"
SHAP_DIR.mkdir(parents=True, exist_ok=True)

np.save(SHAP_DIR / "shap_values.npy", sv.values)
np.save(SHAP_DIR / "shap_base.npy", sv.base_values)
X_test.to_parquet(SHAP_DIR / "X_test.parquet")
y_test.to_frame().to_parquet(SHAP_DIR / "y_test.parquet")

print("saved to:", SHAP_DIR.resolve())

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FIG_DIR  = PROJECT_ROOT / "results" / "figures"
SHAP_DIR = PROJECT_ROOT / "results" / "phase4_shap"
for d in (FIG_DIR, SHAP_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("FIG_DIR :", FIG_DIR.resolve())
print("SHAP_DIR:", SHAP_DIR.resolve())

In [ ]:
import matplotlib.pyplot as plt

shap.plots.bar(sv, max_display=11, show=False)
plt.title("Mean |SHAP| — global feature importance")
plt.tight_layout()
plt.savefig(FIG_DIR / "phase4_shap_bar.png", dpi=150, bbox_inches="tight")
plt.show()

shap.plots.beeswarm(sv, max_display=11, show=False)
plt.tight_layout()
plt.savefig(FIG_DIR / "phase4_shap_beeswarm.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
imp = pd.DataFrame({
    "feature": X_test.columns,
    "impurity": rf_unscaled.feature_importances_,
    "shap_mean_abs": np.abs(sv.values).mean(axis=0),
}).sort_values("shap_mean_abs", ascending=False)
imp["rank_imp"] = imp["impurity"].rank(ascending=False)
imp["rank_shap"] = imp["shap_mean_abs"].rank(ascending=False)
print(imp)

In [ ]:
shap.plots.scatter(sv[:, "alcohol"], color=sv[:, "density"], show=False)
plt.savefig(FIG_DIR / "phase4_shap_dep_alcohol.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
shap.plots.scatter(sv[:, "volatile acidity"], color=sv, show=False)   # auto interaction
plt.savefig(FIG_DIR / "phase4_shap_dep_va.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
shap.plots.scatter(sv[:, "free sulfur dioxide"], color=sv[:, "total sulfur dioxide"], show=False)
plt.savefig(FIG_DIR / "phase4_shap_dep_so2.png", dpi=150, bbox_inches="tight")
plt.show()



In [ ]:
#FEAUTURE_ROLES
FEATURE_ROLES = {
    "levers":               ["free sulfur dioxide", "sulphates", "citric acid"],
    "semi_levers":          ["fixed acidity", "pH", "residual sugar"],
    "constrained_outcomes": ["total sulfur dioxide", "density",
                             "volatile acidity", "alcohol"],
    "context":              ["chlorides"],
}

#single source of truth for column order (matches the dataset)
CANONICAL_FEATURE_ORDER = list(X_train.columns)

#sanity check: FEATURE_ROLES must cover exactly the 11 features, no typos
all_roles = [f for roles in FEATURE_ROLES.values() for f in roles]
assert sorted(all_roles) == sorted(CANONICAL_FEATURE_ORDER), \
    "FEATURE_ROLES does not cover exactly the model features!"
print("FEATURE_ROLES OK:", {k: len(v) for k, v in FEATURE_ROLES.items()})


In [ ]:
#FEAUTURE_ROLES
FEATURE_ROLES = {
    "levers":               ["free sulfur dioxide", "sulphates", "citric acid"],
    "semi_levers":          ["fixed acidity", "pH", "residual sugar"],
    "constrained_outcomes": ["total sulfur dioxide", "density",
                             "volatile acidity", "alcohol"],
    "context":              ["chlorides"],
}


In [ ]:
#logging all of them 
with mlflow.start_run(run_name="rf_final_unscaled"):
    mlflow.set_tag("phase", "4")
    mlflow.set_tag("role", "phase6_source_of_truth")

    mlflow.log_params(rf_best_params)
    mlflow.log_param("scaler", "none (trees are scale-invariant)")
    mlflow.log_metric("test_rmse", float(np.sqrt(mean_squared_error(y_test, pred_unscaled))))
    mlflow.log_metric("test_r2", float(r2_score(y_test, pred_unscaled)))
    mlflow.log_metric("max_abs_delta_vs_scaled", float(np.abs(pred_unscaled - pred_scaled).max()))

    #feature semantics travel WITH the model (same run)
    mlflow.log_dict(FEATURE_ROLES, "feature_roles.json")
    mlflow.log_dict({"canonical_feature_order": CANONICAL_FEATURE_ORDER},
                    "canonical_feature_order.json")
    #split indices so notebook 05 inherits the exact same train/test split
    mlflow.log_dict({"train_idx": X_train.index.tolist(),
                     "test_idx": X_test.index.tolist()},
                    "split_indices.json")

    mlflow.sklearn.log_model(
        rf_unscaled, name="rf_final_unscaled",
        signature=infer_signature(X_train, pred_unscaled),
        input_example=X_train.iloc[:5],
        registered_model_name="wine_quality_rf",
    )
